# 04. Text and bytes

**Tier:** Intermediate. **Why you'd use this:** A file of UTF-8 text is bytes on disk. You need to see the difference between `str` and `bytes`, and to decode with the right encoding.

**After this lesson you can:** encode and decode UTF-8; open files in text versus binary mode; explain why encoding belongs at the boundary.
About 25 minutes. **Uses:** [Paths](01-paths-lesson.ipynb), [Strings](../02-strings/01-indexing-lesson.ipynb)


## Quick review

1. Why open CSV with `newline=""`?
2. What type are DictReader cell values?

<details><summary>Answers</summary>

So csv owns line endings. Strings.
</details>


## The idea

`str` holds Unicode text. `bytes` holds raw octets. `.encode()` turns text into bytes. `.decode()` turns bytes into text. Text mode on files decodes for you. Binary mode returns bytes and does not apply an encoding.

<details><summary>Explain more</summary>

`"é".encode("utf-8")` may be more than one byte. Decoding with the wrong codec raises or produces mojibake.
</details>



## See it work

Encode a string and write bytes to a file.


In [1]:
text = "café"
data = text.encode("utf-8")
# 1. bytes are not str.
print(data)
print(data.decode("utf-8"))

from pathlib import Path

path = Path("cafe.bin")
path.write_bytes(data)
print(path.read_bytes())
path.unlink()


b'caf\xc3\xa9'
café
b'caf\xc3\xa9'


## Check yourself

Answer these before you run the cell.

1. What type does `read_bytes` return?
2. What goes wrong if you decode UTF-8 bytes as a different encoding?


In [2]:
print(type("café".encode("utf-8")).__name__)
print("café".encode("utf-8").decode("utf-8"))


bytes
café


<details><summary>Answers</summary>

`bytes`. You get an error or the wrong characters.
</details>


## Why it works this way

Text in Python is Unicode. Files and sockets are bytes. Encoding is the bridge. UTF-8 is the usual bridge for interoperable text, which is why lessons keep `encoding="utf-8"` visible even when defaults move toward UTF-8. Explicit is still clearer at the boundary. Deep dive: [Numbers and strings](../../internals/05-numbers/01-numbers-strings-lesson.ipynb).

Binary mode (`"rb"`, `"wb"`, `read_bytes`) skips decoding. Use it for images, compressed data, or when you will decode yourself. Mixing text and bytes in one operation raises TypeError on purpose.

Cost of UTF-8 encode/decode is proportional to size. The design reason to separate `str` and `bytes` is to stop silent corruption. Evidence: round-trip encode/decode, and `type` of `read_bytes`. Wrong decoders fail loudly or produce garbage; tests should use known UTF-8 samples.  Mojibake is decoded bytes with the wrong codec. Once you lose the original bytes, repair is guesswork. Keep raw bytes until you know the encoding.

Some codecs cannot represent every Unicode character. Encoding with ASCII raises for `é` unless you pick an error handler. Prefer failing fast over replacing characters in data you must keep. For network protocols, the encoding is part of the protocol specification, not a local preference. Surrogate escapes and replacement characters are for recovery tools, not for ordinary application paths.  BOM markers at the start of some UTF-8 files are another boundary detail. Strip or accept them explicitly when you know a producer adds them. Keep both forms in tests.

That cost shows up when the input grows: each extra unit of work multiplies wall time or memory, so the design choice is not only correctness but whether the approach stays practical under load.


In [3]:
text = "café"
data = text.encode("utf-8")
print(data)
print(list(data))
print(data.decode("utf-8"))
from pathlib import Path

path = Path("cafe.bin")
path.write_bytes(data)
print(path.read_bytes() == data)
path.unlink()


b'caf\xc3\xa9'
[99, 97, 102, 195, 169]
café
True


## When to use it

Use text mode with an encoding for human-readable files. Use bytes mode for opaque binary. Decode at the edge of your program. Don't concatenate `str` and `bytes`. Don't guess encodings for binary formats. Keep raw bytes until the codec is known, then decode once.


## Common mistake

Decoding UTF-8 bytes with the wrong codec and treating the garbage as data.


In [4]:
data = "café".encode("utf-8")
wrong = data.decode("latin-1")
print(wrong)


cafÃ©


Decode with the encoding that produced the bytes.


In [5]:
data = "café".encode("utf-8")
print(data.decode("utf-8"))


café


## Try it

What does `len("é")` versus `len("é".encode("utf-8"))` print, and which one measures code points?


## Recap

- `str` is text. `bytes` is raw data.
- Encode and decode at boundaries with an explicit codec.
- Text mode decodes; binary mode does not.

**Next:** [practice](04-text-and-bytes-practice.ipynb)